In [1]:
import pandas as pd
import nltk
from collections import Counter
from typing import List, Tuple, Dict

nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize


def load_data(filepath: str) -> List[str]:
    """Load Twitter data and return sentences."""
    df = pd.read_csv(filepath, encoding='latin-1', header=None,
                     names=['target', 'ids', 'date', 'flag', 'user', 'text'])
    return df['text'].tolist()


def split_sentences(data: List[str]) -> List[str]:
    """Split data into sentences by newline."""
    return data


def tokenize_sentences(sentences: List[str]) -> List[List[str]]:
    """Tokenize sentences and convert to lowercase."""
    tokenized = []
    for sent in sentences:
        tokens = word_tokenize(sent.lower())
        tokenized.append(tokens)
    return tokenized


def train_test_split(sentences: List[List[str]], test_ratio: float = 0.2) -> Tuple[List[List[str]], List[List[str]]]:
    """Split data into train and test sets."""
    split_idx = int(len(sentences) * (1 - test_ratio))
    return sentences[:split_idx], sentences[split_idx:]


def get_word_counts(train_data: List[List[str]]) -> Counter:
    """Count word frequencies in training data."""
    counts = Counter()
    for sent in train_data:
        counts.update(sent)
    return counts


def get_closed_vocabulary(word_counts: Counter, threshold: int) -> set:
    """Get vocabulary with words appearing >= threshold times."""
    return {word for word, count in word_counts.items() if count >= threshold}


def replace_oov(sentences: List[List[str]], vocabulary: set, unk_token: str = '<unk>') -> List[List[str]]:
    """Replace out-of-vocabulary words with unknown token."""
    replaced = []
    for sent in sentences:
        replaced_sent = [word if word in vocabulary else unk_token for word in sent]
        replaced.append(replaced_sent)
    return replaced


def preprocess(filepath: str, threshold: int = 5, test_ratio: float = 0.2) -> Tuple[List[List[str]], List[List[str]], set]:
    """Complete preprocessing pipeline."""
    data = load_data(filepath)
    sentences = split_sentences(data)
    tokenized = tokenize_sentences(sentences)
    train, test = train_test_split(tokenized, test_ratio)

    word_counts = get_word_counts(train)
    vocab = get_closed_vocabulary(word_counts, threshold)

    train_replaced = replace_oov(train, vocab)
    test_replaced = replace_oov(test, vocab)

    return train_replaced, test_replaced, vocab